# 08c Memory Lane Postprocess v1

이 노트북은 **영상/overlay를 만들지 않는다.** 목적은 08b 이후의 주행 후처리 아이디어를 코드로 읽을 수 있게 구현하는 것이다.

핵심 변화:

- decoder는 고정한다. 입력은 `lanes = [{"points": ..., "conf": ...}, ...]`라고 가정한다.
- 주행 후처리는 `steer_norm`만 계산하지 않고, `mode`, `speed_scale`, `confidence`, `reason`을 함께 출력한다.
- px 단위 threshold는 config에 노출하지 않고 내부에서 이미지 크기 기반으로 자동 계산한다.
- 사람이 만지는 값은 속도, gain, memory strength 정도로 제한한다.

큰 흐름:

```text
decoder lanes
→ lane feature 추출
→ two-lane pair / one-lane reference / lost 판단
→ 이전 stable geometry memory와 비교
→ steer_norm + speed_scale + mode 출력
```

## 1. 고정 geometry contract

아래 값은 12번 실험에서 확정한 학습/추론 좌표계다. 실습 중 수정 대상이 아니다.

- 원본 카메라 프레임: `1296 x 972`
- crop: `y >= 445`
- model input: `800 x 320`
- decoder lane points: 원본 이미지 좌표계의 `(x, y)` points

In [1]:
from __future__ import annotations

import math
from pathlib import Path
import json

import numpy as np

RAW_W = 1296
RAW_H = 972
CUT_HEIGHT = 445
IMAGE_CENTER_X = RAW_W / 2.0

# decoder가 최종 lane points를 만들 때 쓰는 y 샘플 범위.
# 이론상 y=451까지 볼 수 있지만, 조향에 직접 쓰기보다 far 판단에만 쓰는 것이 안전하다.
SAMPLE_Y_MIN = 451
SAMPLE_Y_MAX = 971

## 2. 사람이 이해하는 tuning config

`px` 단위 값은 일부러 넣지 않았다.

- `stable_*`: 두 레인이 안정적으로 보일 때
- `single_*`: 한쪽 레인만 믿을 수 있을 때
- `lost_*`: 레인을 잃었을 때
- `memory_strength`: 이전 안정 방향을 얼마나 고집할지

`speed_scale`은 모터 기본 속도에 곱해질 값이다.

In [2]:
LANE_BEHAVIOR = {
    "name": "memory_lane_postprocess_v1",

    # mode별 속도. 최종 속도는 motor_base_speed * speed_scale.
    "stable_speed_scale": 1.00,
    "single_speed_scale": 0.55,
    "lost_short_speed_scale": 0.35,
    "lost_search_speed_scale": 0.25,

    # 조향 gain. center는 두 레인이 있을 때 도로 중앙 보정, slope는 local 진행방향 보정.
    "center_gain": 1.00,
    "slope_gain": 0.35,
    "single_x_gain": 0.25,
    "single_slope_gain": 0.55,

    # 이전 stable direction을 얼마나 고집할지. 0이면 현재 frame을 많이 믿고, 1이면 memory를 많이 믿는다.
    "memory_strength": 0.75,
    "stable_update_alpha": 0.20,

    # lane을 잃은 직후에는 이전 조향을 유지하고, 오래 잃으면 조금 더 강하게 탐색한다.
    "lost_short_frames": 5,
    "lost_search_boost": 1.15,

    # 최종 steer clamp.
    "max_steer_norm": 0.70,
}

## 3. 내부 threshold 자동 계산

여기서만 px 단위 값이 등장한다. 현장에서 직접 수정하지 않는 내부 규칙이다.

- `near/mid/far_y`: 조향과 안정 직선 판단에 쓰는 y 위치
- `pair_gap_*`: 두 lane이 정상 pair인지 보는 내부 범위
- `jump_*`: 현재 geometry가 이전 stable memory에서 너무 튀었는지 보는 내부 범위

In [3]:
def clamp(v, lo, hi):
    return max(lo, min(hi, float(v)))


def derive_internal_thresholds(cfg):
    memory = clamp(cfg["memory_strength"], 0.0, 1.0)

    return {
        # 조향은 아래쪽을 중심으로, far는 복귀/안정성 판단에만 보조로 쓴다.
        "near_y": RAW_H * 0.96,
        "mid_y": RAW_H * 0.84,
        "far_y": RAW_H * 0.68,

        # lane feature 유효성. 이미지 크기 기반 고정 비율.
        "min_points": 4,
        "min_y_span_px": RAW_H * 0.06,
        "max_interp_gap_px": RAW_H * 0.13,

        # 두 lane pair 정상 범위. 너무 좁으면 중복 lane, 너무 넓으면 잘못된 pair 가능성.
        "pair_gap_min_px": RAW_W * 0.22,
        "pair_gap_max_px": RAW_W * 0.92,
        "expected_half_gap_fallback": RAW_W * 0.33,

        # memory가 강할수록 jump를 더 엄격히 본다.
        "center_jump_norm": 0.20 - 0.10 * memory,
        "slope_jump": 0.75 - 0.25 * memory,

        # pair scoring 가중치. 외부 튜닝값이 아니라 선택 기준이다.
        "pair_center_weight": 1.0,
        "pair_gap_weight": 0.4,
        "pair_slope_diff_weight": 0.8,
        "pair_memory_weight": 0.8,
    }


INTERNAL = derive_internal_thresholds(LANE_BEHAVIOR)
INTERNAL

{'near_y': 933.12,
 'mid_y': 816.48,
 'far_y': 660.96,
 'min_points': 4,
 'min_y_span_px': 58.32,
 'max_interp_gap_px': 126.36,
 'pair_gap_min_px': 285.12,
 'pair_gap_max_px': 1192.3200000000002,
 'expected_half_gap_fallback': 427.68,
 'center_jump_norm': 0.125,
 'slope_jump': 0.5625,
 'pair_center_weight': 1.0,
 'pair_gap_weight': 0.4,
 'pair_slope_diff_weight': 0.8,
 'pair_memory_weight': 0.8}

## 4. Lane feature 추출

decoder output의 lane points에서 `x_near`, `x_mid`, `x_far`, `slope`를 뽑는다.

- `slope > 0`: 위쪽으로 갈수록 오른쪽으로 향하는 lane
- `slope < 0`: 위쪽으로 갈수록 왼쪽으로 향하는 lane

이 함수는 lane이 충분히 길지 않거나, 필요한 y 위치에서 x를 얻기 어렵다면 `None`을 반환한다.

In [4]:
def interp_or_nearest_x(points, y, max_gap_px):
    pts = np.asarray(points, dtype=np.float32)
    if len(pts) == 0:
        return None

    xs = pts[:, 0]
    ys = pts[:, 1]
    order = np.argsort(ys)
    ys = ys[order]
    xs = xs[order]

    if float(ys.min()) <= y <= float(ys.max()) and len(pts) >= 2:
        return float(np.interp(y, ys, xs))

    nearest = int(np.argmin(np.abs(ys - y)))
    if abs(float(ys[nearest]) - float(y)) <= float(max_gap_px):
        return float(xs[nearest])
    return None


def lane_feature(lane, internal):
    pts = np.asarray(lane.get("points", []), dtype=np.float32)
    if len(pts) < int(internal["min_points"]):
        return None

    y_span = float(pts[:, 1].max() - pts[:, 1].min())
    if y_span < float(internal["min_y_span_px"]):
        return None

    x_near = interp_or_nearest_x(pts, internal["near_y"], internal["max_interp_gap_px"])
    x_mid = interp_or_nearest_x(pts, internal["mid_y"], internal["max_interp_gap_px"])
    x_far = interp_or_nearest_x(pts, internal["far_y"], internal["max_interp_gap_px"])
    if x_near is None or x_mid is None or x_far is None:
        return None

    # near~mid는 실제 조향, near~far는 멀리 안정 방향 판단에 더 가깝다.
    slope_near = (x_mid - x_near) / max(1.0, internal["near_y"] - internal["mid_y"])
    slope_far = (x_far - x_near) / max(1.0, internal["near_y"] - internal["far_y"])
    slope = 0.70 * slope_near + 0.30 * slope_far

    return {
        "x_near": float(x_near),
        "x_mid": float(x_mid),
        "x_far": float(x_far),
        "slope": float(slope),
        "slope_near": float(slope_near),
        "slope_far": float(slope_far),
        "conf": float(lane.get("conf", lane.get("score", 1.0))),
        "y_span": y_span,
        "raw_lane": lane,
    }

## 5. Memory

08b는 `last_steer_norm`만 기억했다. 08c는 안정적인 두 레인이 보였을 때의 geometry를 기억한다.

기억하는 값:

- 안정적인 lane center
- 안정적인 left/right lane 위치
- 안정적인 slope 평균
- 최근 steer
- lost frame 수

In [5]:
def init_memory():
    return {
        "has_stable": False,
        "stable_center_x": IMAGE_CENTER_X,
        "stable_left_x": IMAGE_CENTER_X - INTERNAL["expected_half_gap_fallback"],
        "stable_right_x": IMAGE_CENTER_X + INTERNAL["expected_half_gap_fallback"],
        "stable_half_gap": INTERNAL["expected_half_gap_fallback"],
        "stable_slope": 0.0,
        "last_steer_norm": 0.0,
        "last_mode": "init",
        "lost_frames": 0,
    }


def update_stable_memory(memory, pair_geom, cfg):
    alpha = clamp(cfg["stable_update_alpha"], 0.0, 1.0)
    memory["has_stable"] = True
    memory["stable_center_x"] = (1.0 - alpha) * memory["stable_center_x"] + alpha * pair_geom["center_x"]
    memory["stable_left_x"] = (1.0 - alpha) * memory["stable_left_x"] + alpha * pair_geom["left"]["x_mid"]
    memory["stable_right_x"] = (1.0 - alpha) * memory["stable_right_x"] + alpha * pair_geom["right"]["x_mid"]
    memory["stable_half_gap"] = (1.0 - alpha) * memory["stable_half_gap"] + alpha * (0.5 * pair_geom["gap_px"])
    memory["stable_slope"] = (1.0 - alpha) * memory["stable_slope"] + alpha * pair_geom["slope"]

## 6. 두 lane pair 선택

두 lane이 여러 개 보일 수 있으므로 가능한 pair들을 점수화한다.

좋은 pair란:

- lane 사이 간격이 정상적이고
- 중심이 화면 중앙 근처이며
- 두 lane 기울기가 서로 크게 다르지 않고
- 이전 stable memory와 크게 튀지 않는 pair

In [6]:
def pair_geometry(a, b):
    left, right = (a, b) if a["x_mid"] <= b["x_mid"] else (b, a)
    gap = right["x_mid"] - left["x_mid"]
    center_x = 0.5 * (left["x_mid"] + right["x_mid"])
    slope = 0.5 * (left["slope"] + right["slope"])
    center_error = (center_x - IMAGE_CENTER_X) / IMAGE_CENTER_X
    return {
        "left": left,
        "right": right,
        "gap_px": float(gap),
        "center_x": float(center_x),
        "center_error": float(center_error),
        "slope": float(slope),
        "slope_diff": float(abs(left["slope"] - right["slope"])),
        "mean_conf": float(0.5 * (left["conf"] + right["conf"])),
    }


def score_pair(g, memory, internal):
    gap_mid = 0.5 * (internal["pair_gap_min_px"] + internal["pair_gap_max_px"])
    gap_norm = abs(g["gap_px"] - gap_mid) / max(1.0, gap_mid)
    score = 0.0
    score += internal["pair_center_weight"] * abs(g["center_error"])
    score += internal["pair_gap_weight"] * gap_norm
    score += internal["pair_slope_diff_weight"] * abs(g["slope_diff"])
    score -= 0.15 * g["mean_conf"]

    if memory["has_stable"]:
        center_jump = abs((g["center_x"] - memory["stable_center_x"]) / IMAGE_CENTER_X)
        slope_jump = abs(g["slope"] - memory["stable_slope"])
        score += internal["pair_memory_weight"] * (center_jump + slope_jump)
    return float(score)


def best_pair(features, memory, internal):
    best = None
    best_score = float("inf")
    for i in range(len(features)):
        for j in range(i + 1, len(features)):
            g = pair_geometry(features[i], features[j])
            if not (internal["pair_gap_min_px"] <= g["gap_px"] <= internal["pair_gap_max_px"]):
                continue
            s = score_pair(g, memory, internal)
            if s < best_score:
                best = g
                best_score = s
    if best is not None:
        best["pair_score"] = best_score
    return best


def pair_is_jumpy(pair, memory, internal):
    if pair is None or not memory["has_stable"]:
        return False
    center_jump = abs((pair["center_x"] - memory["stable_center_x"]) / IMAGE_CENTER_X)
    slope_jump = abs(pair["slope"] - memory["stable_slope"])
    return center_jump > internal["center_jump_norm"] or slope_jump > internal["slope_jump"]

## 7. 한쪽 lane reference 선택

pair가 없거나 pair가 jumpy하면 한쪽 lane을 reference로 쓴다.

우선순위:

1. 이전 stable left/right 위치와 가장 자연스럽게 이어지는 lane
2. 이전 stable slope와 가까운 lane
3. 그래도 없으면 화면 중앙과 가까운 lane

한쪽 lane이 이전 left/right 중 무엇과 이어지는지 알 수 있으면, 이전 half gap을 이용해 임시 center를 만든다.

In [7]:
def choose_single_reference(features, memory):
    if not features:
        return None

    best = None
    best_score = float("inf")
    for f in features:
        if memory["has_stable"]:
            left_dist = abs(f["x_mid"] - memory["stable_left_x"])
            right_dist = abs(f["x_mid"] - memory["stable_right_x"])
            role = "left" if left_dist <= right_dist else "right"
            lane_dist = min(left_dist, right_dist) / IMAGE_CENTER_X
            slope_dist = abs(f["slope"] - memory["stable_slope"])
            score = lane_dist + 0.5 * slope_dist - 0.1 * f["conf"]
        else:
            role = "unknown"
            score = abs(f["x_mid"] - IMAGE_CENTER_X) / IMAGE_CENTER_X - 0.1 * f["conf"]

        if score < best_score:
            best = dict(f)
            best["role"] = role
            best["single_score"] = float(score)
            best_score = score
    return best


def single_center_error(single, memory):
    if single is None:
        return 0.0
    if memory["has_stable"] and single["role"] == "left":
        center_x = single["x_mid"] + memory["stable_half_gap"]
    elif memory["has_stable"] and single["role"] == "right":
        center_x = single["x_mid"] - memory["stable_half_gap"]
    else:
        # role이 없으면 x 위치를 강하게 믿지 않는다. slope가 주 역할을 하게 둔다.
        center_x = IMAGE_CENTER_X
    return float((center_x - IMAGE_CENTER_X) / IMAGE_CENTER_X)

## 8. Geometry → drive mode

mode는 일부러 적게 둔다.

- `both_stable`: 두 lane pair가 안정적이다.
- `single_follow`: 한쪽 lane만 reference로 사용한다. pair가 jumpy인 경우도 여기로 떨어진다.
- `lost_short`: lane을 잠깐 잃었다.
- `lost_search`: lane을 오래 잃었다. 이전 방향으로 더 강하게 탐색한다.

In [8]:
def build_geometry(lanes, memory, cfg, internal):
    features = [f for f in (lane_feature(lane, internal) for lane in lanes) if f is not None]
    pair = best_pair(features, memory, internal)

    if pair is not None and not pair_is_jumpy(pair, memory, internal):
        return {
            "mode": "both_stable",
            "features": features,
            "pair": pair,
            "single": None,
            "center_error": pair["center_error"],
            "slope": pair["slope"],
            "confidence": min(1.0, pair["mean_conf"]),
            "reason": "valid pair: use center_error + averaged slope",
        }

    if features:
        single = choose_single_reference(features, memory)
        center_error = single_center_error(single, memory)
        slope = single["slope"]
        if memory["has_stable"]:
            m = clamp(cfg["memory_strength"], 0.0, 1.0)
            # 한쪽 lane에서는 현재 slope를 바로 믿지 않고 이전 stable slope와 섞는다.
            slope = m * memory["stable_slope"] + (1.0 - m) * slope
        reason = "single feature: follow reference lane"
        if pair is not None:
            reason = "pair exists but jumpy: fall back to single reference"
        return {
            "mode": "single_follow",
            "features": features,
            "pair": pair,
            "single": single,
            "center_error": float(center_error),
            "slope": float(slope),
            "confidence": float(single["conf"]),
            "reason": reason,
        }

    lost_frames = int(memory["lost_frames"]) + 1
    mode = "lost_short" if lost_frames <= int(cfg["lost_short_frames"]) else "lost_search"
    return {
        "mode": mode,
        "features": [],
        "pair": None,
        "single": None,
        "center_error": 0.0,
        "slope": memory["stable_slope"] if memory["has_stable"] else 0.0,
        "confidence": 0.0,
        "reason": "no valid lane feature",
    }

## 9. Drive command 계산

최종 출력은 `steer_norm`과 `speed_scale`이다.

모터는 나중에 다음처럼 단순히 받으면 된다.

```python
speed = motor_base_speed * lane_drive["speed_scale"]
turn  = motor_steer_to_turn * lane_drive["steer_norm"]
```

In [9]:
def clip_steer(v, cfg):
    return clamp(v, -cfg["max_steer_norm"], cfg["max_steer_norm"])


def target_from_geometry(geom, memory, cfg):
    mode = geom["mode"]

    if mode == "both_stable":
        steer = cfg["center_gain"] * geom["center_error"] + cfg["slope_gain"] * geom["slope"]
        speed_scale = cfg["stable_speed_scale"]
    elif mode == "single_follow":
        steer = cfg["single_x_gain"] * geom["center_error"] + cfg["single_slope_gain"] * geom["slope"]
        speed_scale = cfg["single_speed_scale"]
    elif mode == "lost_short":
        steer = memory["last_steer_norm"]
        speed_scale = cfg["lost_short_speed_scale"]
    else:  # lost_search
        steer = memory["last_steer_norm"] * cfg["lost_search_boost"]
        speed_scale = cfg["lost_search_speed_scale"]

    return clip_steer(steer, cfg), float(speed_scale)


def smooth_steer(target, memory, mode, cfg):
    # stable은 빠르게 반응, single은 memory를 조금 더 고집, lost는 별도 smoothing 없이 유지/강화.
    if mode == "both_stable":
        alpha = 0.70
    elif mode == "single_follow":
        alpha = 1.0 - 0.55 * clamp(cfg["memory_strength"], 0.0, 1.0)
    else:
        alpha = 1.0
    return clip_steer(alpha * target + (1.0 - alpha) * memory["last_steer_norm"], cfg)


def update_drive(lanes, memory, cfg=LANE_BEHAVIOR):
    internal = derive_internal_thresholds(cfg)
    geom = build_geometry(lanes, memory, cfg, internal)
    target, speed_scale = target_from_geometry(geom, memory, cfg)
    steer = smooth_steer(target, memory, geom["mode"], cfg)

    if geom["mode"] == "both_stable":
        update_stable_memory(memory, geom["pair"], cfg)
        memory["lost_frames"] = 0
    elif geom["mode"] == "single_follow":
        memory["lost_frames"] = 0
        # single에서는 stable center는 갱신하지 않는다. slope만 아주 천천히 따라가게 한다.
        if memory["has_stable"]:
            alpha = 0.05
            memory["stable_slope"] = (1.0 - alpha) * memory["stable_slope"] + alpha * geom["slope"]
    else:
        memory["lost_frames"] += 1

    memory["last_steer_norm"] = steer
    memory["last_mode"] = geom["mode"]

    return {
        "mode": geom["mode"],
        "steer_norm": float(steer),
        "speed_scale": float(speed_scale),
        "center_error": float(geom["center_error"]),
        "slope": float(geom["slope"]),
        "confidence": float(geom["confidence"]),
        "stable_forward_found": bool(geom["mode"] == "both_stable"),
        "lane_count": int(len(lanes)),
        "feature_count": int(len(geom["features"])),
        "lost_frames": int(memory["lost_frames"]),
        "reason": geom["reason"],
    }

## 10. 읽기용 synthetic smoke test

실제 모델 없이도 함수의 동작을 읽을 수 있도록 단순 lane points를 만든다. 이 셀은 algorithm sanity check용이며, 최종 성능 판단용이 아니다.

In [10]:
def make_line_lane(x_bottom, x_far, conf=0.9):
    ys = np.linspace(RAW_H * 0.98, RAW_H * 0.62, 16, dtype=np.float32)
    xs = np.linspace(x_bottom, x_far, 16, dtype=np.float32)
    return {"points": np.stack([xs, ys], axis=1), "conf": conf}


memory = init_memory()
synthetic_frames = [
    # two stable lanes
    [make_line_lane(320, 430), make_line_lane(980, 1080)],
    [make_line_lane(330, 440), make_line_lane(990, 1090)],
    # one lane remains; should use memory + single reference
    [make_line_lane(340, 460)],
    # lost short
    [],
    # lost search after repeated lost frames
    [], [], [], [], [], [],
]

rows = []
for idx, lanes in enumerate(synthetic_frames):
    out = update_drive(lanes, memory, LANE_BEHAVIOR)
    rows.append({"frame": idx, **out})

rows

[{'frame': 0,
  'mode': 'both_stable',
  'steer_norm': 0.1197873386777115,
  'speed_scale': 1.0,
  'center_error': 0.06610081007635066,
  'slope': 0.3000684556092492,
  'confidence': 0.9,
  'stable_forward_found': True,
  'lane_count': 2,
  'feature_count': 2,
  'lost_frames': 0,
  'reason': 'valid pair: use center_error + averaged slope'},
 {'frame': 1,
  'mode': 'both_stable',
  'steer_norm': 0.16652600941682744,
  'speed_scale': 1.0,
  'center_error': 0.08153290884178276,
  'slope': 0.3000684556092492,
  'confidence': 0.9,
  'stable_forward_found': True,
  'lane_count': 2,
  'feature_count': 2,
  'lost_frames': 0,
  'reason': 'valid pair: use center_error + averaged slope'},
 {'frame': 2,
  'mode': 'single_follow',
  'steer_norm': 0.15214871663290602,
  'speed_scale': 0.55,
  'center_error': 0.20136089469753102,
  'slope': 0.16675236004569388,
  'confidence': 0.9,
  'stable_forward_found': False,
  'lane_count': 1,
  'feature_count': 1,
  'lost_frames': 0,
  'reason': 'single featur

## 11. 산출물 저장

이 노트북은 아직 runtime에 바로 반영하지 않는다. 먼저 코드를 읽고, 원칙이 맞는지 검토한다.

In [11]:
OUT_DIR = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\08c_memory_lane_postprocess_v1")
CONFIG_DIR = OUT_DIR / "config"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_DIR / "lane_behavior_v3_config.json", "w", encoding="utf-8") as f:
    json.dump(LANE_BEHAVIOR, f, ensure_ascii=False, indent=2)

print("wrote", CONFIG_DIR / "lane_behavior_v3_config.json")

wrote ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\08c_memory_lane_postprocess_v1\config\lane_behavior_v3_config.json


## 12. 다음 판단 기준

이 구현은 아직 최종안이 아니다. 읽으면서 봐야 할 지점은 다음이다.

1. `single_follow`에서 x 좌표를 얼마나 믿을 것인가
2. lost에서 이전 steer를 강화하는 것이 실제 코너 복구에 도움이 되는가
3. `speed_scale`을 lane 후처리에서 내는 구조가 runtime motor 구조와 잘 맞는가
4. mode를 더 늘리지 않고도 현장 튜닝이 가능한가

## 13. field3 sequence replay video

08c는 원래 코드 읽기용 구현이지만, 로직이 실제 field3 sequence에서 어떻게 움직이는지는 영상으로 보는 편이 빠르다.

여기서는 모델을 다시 추론하지 않고, 10번 Pi runtime validation package에 저장된 **field3 sequence decoded lane**을 그대로 사용한다. 즉 비교 대상은 다음과 같다.

```text
10번 package decoded lane points
    → 08c memory lane postprocess
    → steer_norm / speed_scale / mode
    → replay video
```

영상에서 초록 선은 decoder lane, 자홍색 화살표는 최종 `steer_norm`, 노란 화살표는 현재 geometry slope 방향을 뜻한다.  
이 셀은 성능 확정용이 아니라, **로직이 어떤 mode로 판단하고 얼마나 강하게 조향하는지 읽기 위한 sanity replay**다.


In [ ]:
import csv
import cv2
from collections import Counter

BASE = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild")
REVIEW_ROOT = BASE / "review_outputs" / "08c_memory_lane_postprocess_v1"
REVIEW_ROOT.mkdir(parents=True, exist_ok=True)

PKG10 = BASE / "review_outputs" / "10_pi_runtime_latency_sequence_validation_v1" / "pkg"
RECORDS_CSV = PKG10 / "t" / "records_manifest.csv"
DECODED_JSONL = PKG10 / "r" / "ref_decoded.jsonl"
VIDEO_DIR = REVIEW_ROOT / "videos"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR = REVIEW_ROOT / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

FIELD3_VIDEO_PATH = VIDEO_DIR / "field3_08c_memory_lane_postprocess_v1.mp4"
FIELD3_SEQUENCE_CSV = TABLE_DIR / "field3_08c_memory_lane_postprocess_v1.csv"

print("records:", RECORDS_CSV)
print("decoded:", DECODED_JSONL)
print("video:", FIELD3_VIDEO_PATH)


## 14. field3 replay helpers

이 helper들은 최종 runtime 로직이 아니라 노트북 검수용이다.  
영상에서는 원본 frame 위에 다음 정보만 그린다.

- green: decoder가 반환한 lane polyline
- gray vertical: 화면 중앙 기준선
- cyan horizontal: 08c가 보는 near/mid/far y 위치
- magenta arrow: 최종 steer command
- yellow arrow: lane geometry에서 계산된 slope 방향
- text: mode, steer, speed scale, center error, slope, reason


In [ ]:
def imread_bgr_unicode(path):
    path = Path(path)
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return img


def load_field3_sequence_records(limit=None):
    rows = []
    with RECORDS_CSV.open("r", encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            if row["set"] == "field3" and row["role"] == "sequence":
                row["order"] = int(row["order"])
                rows.append(row)
    rows.sort(key=lambda r: r["order"])
    if limit is not None:
        rows = rows[: int(limit)]
    return rows


def load_decoded_lanes_by_key():
    out = {}
    with DECODED_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            out[obj["key"]] = obj.get("lanes", [])
    return out


def lanes_from_jsonable(lanes_json):
    lanes = []
    for lane in lanes_json:
        pts = np.asarray(lane.get("points", []), dtype=np.float32)
        if pts.ndim != 2 or pts.shape[1] != 2 or len(pts) < 2:
            continue
        lanes.append({"points": pts, "conf": float(lane.get("conf", 0.0))})
    return lanes


def draw_lane_polyline(img, points, color=(0, 220, 80), thickness=3):
    pts = np.asarray(points, dtype=np.float32)
    valid = np.isfinite(pts).all(axis=1)
    pts = pts[valid]
    if len(pts) < 2:
        return
    pts_i = np.round(pts).astype(np.int32).reshape(-1, 1, 2)
    cv2.polylines(img, [pts_i], False, color, thickness, cv2.LINE_AA)


def draw_text_lines(img, lines, x=24, y=34, line_h=28):
    pad = 10
    width = max(520, max((len(s) for s in lines), default=0) * 13)
    height = line_h * len(lines) + pad * 2
    panel = img.copy()
    cv2.rectangle(panel, (x - pad, y - 24), (x - pad + width, y - 24 + height), (0, 0, 0), -1)
    cv2.addWeighted(panel, 0.55, img, 0.45, 0, img)
    for i, text in enumerate(lines):
        cv2.putText(img, text, (x, y + i * line_h), cv2.FONT_HERSHEY_SIMPLEX, 0.72, (245, 245, 245), 2, cv2.LINE_AA)


def draw_08c_replay_frame(bgr, lanes, drive, frame_index, cfg=LANE_BEHAVIOR, scale_width=960):
    out = bgr.copy()
    internal = derive_internal_thresholds(cfg)

    for lane in lanes:
        draw_lane_polyline(out, lane["points"], color=(0, 220, 80), thickness=4)

    # 08c가 읽는 y anchor들. near/mid/far가 너무 아래쪽인지 보는 용도다.
    for name, color in [("far", (255, 220, 0)), ("mid", (255, 180, 0)), ("near", (255, 140, 0))]:
        y = int(round(internal[f"{name}_y"]))
        cv2.line(out, (0, y), (RAW_W - 1, y), color, 1, cv2.LINE_AA)
        cv2.putText(out, name, (12, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)

    # 화면 중앙 기준선.
    cv2.line(out, (int(IMAGE_CENTER_X), int(CUT_HEIGHT)), (int(IMAGE_CENTER_X), RAW_H - 1), (200, 200, 200), 2, cv2.LINE_AA)

    base = (int(IMAGE_CENTER_X), RAW_H - 35)
    far_y = int(round(internal["far_y"]))
    mid_y = int(round(internal["mid_y"]))

    steer = float(drive["steer_norm"])
    steer_tip = (int(round(IMAGE_CENTER_X + steer * 420.0)), far_y)
    cv2.arrowedLine(out, base, steer_tip, (255, 0, 220), 5, cv2.LINE_AA, tipLength=0.20)

    slope = float(drive.get("slope", 0.0))
    slope_tip = (int(round(IMAGE_CENTER_X + slope * 130.0)), mid_y)
    cv2.arrowedLine(out, base, slope_tip, (0, 220, 255), 3, cv2.LINE_AA, tipLength=0.18)

    center_error = float(drive.get("center_error", 0.0))
    if np.isfinite(center_error):
        center_x = int(round(IMAGE_CENTER_X + center_error * (RAW_W / 2.0)))
        cv2.circle(out, (center_x, mid_y), 9, (0, 170, 255), -1, cv2.LINE_AA)
        cv2.line(out, (int(IMAGE_CENTER_X), mid_y), (center_x, mid_y), (0, 170, 255), 3, cv2.LINE_AA)

    mode = str(drive["mode"])
    mode_color = {
        "both_stable": (80, 220, 80),
        "single_follow": (0, 220, 255),
        "lost_short": (0, 165, 255),
        "lost_search": (0, 60, 255),
    }.get(mode, (255, 255, 255))
    cv2.circle(out, (RAW_W - 34, 34), 16, mode_color, -1, cv2.LINE_AA)

    lines = [
        f"{frame_index:04d} mode={mode} lanes={drive['lane_count']} features={drive['feature_count']}",
        f"steer={drive['steer_norm']:+.3f} speed_scale={drive['speed_scale']:.2f} conf={drive['confidence']:.2f}",
        f"center={drive['center_error']:+.3f} slope={drive['slope']:+.3f} lost={drive['lost_frames']}",
        f"reason={drive['reason']}",
    ]
    draw_text_lines(out, lines)

    if scale_width is not None and scale_width > 0 and out.shape[1] != scale_width:
        scale = float(scale_width) / float(out.shape[1])
        out = cv2.resize(out, (scale_width, int(round(out.shape[0] * scale))), interpolation=cv2.INTER_AREA)
    return out

print("field3 replay helpers ready")


## 15. field3 replay video 생성

아래 셀을 실행하면 240-frame field3 sequence 전체에 대해 08c 후처리를 순차 적용한다.  
중요한 점은 memory가 frame 순서대로 누적된다는 것이다. 그래서 단일 이미지 overlay보다 실제 주행 후처리 흐름에 더 가깝다.


In [ ]:
def generate_field3_08c_video(limit=None, fps=12.0):
    records = load_field3_sequence_records(limit=limit)
    decoded_by_key = load_decoded_lanes_by_key()
    memory = init_memory()
    writer = None
    rows = []

    for idx, rec in enumerate(records):
        bgr = imread_bgr_unicode(PKG10 / rec["image_rel"])
        lanes = lanes_from_jsonable(decoded_by_key.get(rec["key"], []))
        drive = update_drive(lanes, memory, LANE_BEHAVIOR)
        frame = draw_08c_replay_frame(bgr, lanes, drive, idx, LANE_BEHAVIOR, scale_width=960)

        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(str(FIELD3_VIDEO_PATH), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
            assert writer.isOpened(), FIELD3_VIDEO_PATH

        writer.write(frame)
        rows.append({
            "frame_index": idx,
            "key": rec["key"],
            "source_name": rec["source_name"],
            "mode": drive["mode"],
            "lane_count": drive["lane_count"],
            "feature_count": drive["feature_count"],
            "steer_norm": float(drive["steer_norm"]),
            "speed_scale": float(drive["speed_scale"]),
            "center_error": float(drive["center_error"]),
            "slope": float(drive["slope"]),
            "confidence": float(drive["confidence"]),
            "lost_frames": int(drive["lost_frames"]),
            "reason": drive["reason"],
        })

    if writer is not None:
        writer.release()

    with FIELD3_SEQUENCE_CSV.open("w", encoding="utf-8-sig", newline="") as f:
        writer_csv = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["frame_index"])
        writer_csv.writeheader()
        writer_csv.writerows(rows)

    mode_counts = Counter(row["mode"] for row in rows)
    steer_abs = np.asarray([abs(row["steer_norm"]) for row in rows], dtype=np.float32)
    speed = np.asarray([row["speed_scale"] for row in rows], dtype=np.float32)
    summary = {
        "frames": len(rows),
        "video": str(FIELD3_VIDEO_PATH),
        "table": str(FIELD3_SEQUENCE_CSV),
        "mode_counts": dict(mode_counts),
        "mean_abs_steer": float(steer_abs.mean()) if len(steer_abs) else None,
        "p90_abs_steer": float(np.percentile(steer_abs, 90)) if len(steer_abs) else None,
        "mean_speed_scale": float(speed.mean()) if len(speed) else None,
        "min_speed_scale": float(speed.min()) if len(speed) else None,
    }
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    return summary

field3_08c_video_summary = generate_field3_08c_video(limit=None, fps=12.0)
